# Volume 13, Chapter 9: Bounded gaps between primes (Python companion)

Exercises first, solutions at the end. We check Dirichlet's simplex integrals, rebuild Maynard's Gram matrices in exact rational arithmetic, certify $M_5>2$ and $M_{105}>4$, and test the probabilistic construction of Proposition 9.3. The computer science application: **ill-conditioned generalized eigenvalue problems**, where double precision silently fails and multi-precision arithmetic plus an exact check rescues the result. Requires `numpy` and `mpmath`.

In [ ]:
import math, time
from fractions import Fraction as Fr
from functools import lru_cache
from collections import Counter
import numpy as np
import mpmath as mp

def partitions(n, maxpart=None, maxlen=None):
    if maxpart is None: maxpart = n
    if n == 0: yield (); return
    if maxlen == 0: return
    for p in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - p, p, None if maxlen is None else maxlen - 1):
            yield (p,) + rest

@lru_cache(None)
def sym_int(k, a, b):
    # <a,b>_k = integral over the simplex R_k of (1-P1)^a P2^b  (Lemma 9.3)
    tot = Fr(0)
    for lam in partitions(b, maxlen=k):
        placements = math.factorial(k) // math.factorial(k - len(lam))
        for v in Counter(lam).values(): placements //= math.factorial(v)
        term = math.factorial(b)
        for x in lam: term = term // math.factorial(x) * math.factorial(2 * x)
        tot += placements * term
    return tot * math.factorial(a) / math.factorial(k + a + 2 * b)

def basis(d):
    return [(a, b) for a in range(d + 1) for b in range(d + 1) if a + 2 * b <= d]

def inner(a, b):
    # int_0^u (u-t)^a (Q+t^2)^b dt as a combination of (1-P1')^a' P2'^b'
    return [(Fr(math.comb(b, j) * math.factorial(a) * math.factorial(2 * j), math.factorial(a + 2 * j + 1)),
             a + 2 * j + 1, b - j) for j in range(b + 1)]

def gram(k, d):
    B = basis(d); n = len(B)
    I = [[sym_int(k, B[i][0] + B[j][0], B[i][1] + B[j][1]) for j in range(n)] for i in range(n)]
    J = [[k * sum(c1 * c2 * sym_int(k - 1, a1 + a2, b1 + b2)
                  for c1, a1, b1 in inner(*B[i]) for c2, a2, b2 in inner(*B[j]))
          for j in range(n)] for i in range(n)]
    return B, I, J

def quad(c, M):
    n = len(c)
    return sum(c[i] * c[j] * M[i][j] for i in range(n) for j in range(n))

## Exercises

**P9.1 (★) Dirichlet's integral.** Check Lemma 9.2 for a few exponents by Monte Carlo: sample uniform points of $\mathcal R_k$ (how?), and compare the sample mean of $(1-\sum t_i)^a\prod t_i^{c_i}$ times $\mathrm{vol}(\mathcal R_k)=\frac1{k!}$ with the formula.

**P9.2 (★★) Maynard's $M_5$.** Build the Gram matrices of Lemma 9.3 for $k=5$ in exact rational arithmetic. Check that Maynard's polynomial $F=(1-P_1)P_2+\frac7{10}(1-P_1)^2+\frac1{14}P_2-\frac3{14}(1-P_1)$ gives $I_5=\frac{29509}{1222452000}$ and the ratio $\frac{1417255}{708216}$. Then find the best ratio in degree $d=1,\dots,8$, and the best ratio for $k=2,\dots,10$ in degree $6$; compare with $\frac{k\log k}{k-1}$.

**P9.3 (★★) Certifying $M_{105}>4$.** With the $42$ monomials of degree at most $11$, compute the top generalized eigenvalue for $k=105$ in multi-precision, round the eigenvector to rationals, and check in exact arithmetic that the ratio exceeds $4$. How many significant digits must the rounded coefficients keep?

**P9.4 (★★) The product construction.** For the function $F=\prod g(kt_i)$, $g(t)=\frac1{1+At}$ on $[0,T]$, the proof of Proposition 9.3 gives the bound $\frac{\mathbb P(X_1+\cdots+X_{k-1}\le k-T)}{\gamma}$. Estimate it by simulation, optimize over $A$ and $T$ for several $k$, and find roughly the least $k$ for which it exceeds $4$. Compare with Maynard's choice of $A$ and $T$, with $\log k-2\log\log k-2$, and with the polynomial values.

**P9.5 (★★) Ill-conditioning (computer science).** Repeat P9.3 in double precision with `numpy`. Compute the condition number of the Gram matrix $I$ for $k=5,20,50,105$ (in multi-precision). Explain why Cholesky in double precision fails, and why the final exact check makes the answer trustworthy no matter how it was found.

## Solutions

### P9.1

In [ ]:
rng = np.random.default_rng(0)
def simplex_points(k, n):
    # uniform on R_k: k+1 i.i.d. exponentials, normalized, drop the last coordinate
    E = rng.exponential(size=(n, k + 1))
    return (E / E.sum(axis=1, keepdims=True))[:, :k]

for k, a, cs in [(2, 1, (1, 0)), (3, 2, (1, 1, 0)), (5, 0, (2, 0, 0, 1, 0)), (5, 3, (0,) * 5)]:
    t = simplex_points(k, 10 ** 6)
    vals = (1 - t.sum(axis=1)) ** a * np.prod(t ** np.array(cs), axis=1)
    mc = vals.mean() / math.factorial(k)
    exact = Fr(math.factorial(a) * math.prod(math.factorial(c) for c in cs), math.factorial(k + a + sum(cs)))
    print(f"k={k} a={a} c={cs}:  Monte Carlo {mc:.6e}   formula {float(exact):.6e} = {exact}")

A uniform point of $\mathcal R_k$: take $k+1$ independent exponential variables, divide by their sum (a uniform point of the $k$-dimensional standard simplex in $\mathbb R^{k+1}$, the Dirichlet$(1,\dots,1)$ law) and drop the last coordinate. The Monte Carlo values agree with Lemma 9.2 to three or four digits, as expected from $10^6$ samples. Notice that the same Dirichlet law, with general parameters, is exactly what Lemma 9.2 normalizes: $\frac{a!\,\prod c_i!}{(k+a+\sum c_i)!}$ is its normalizing constant.

### P9.2

In [ ]:
B, I, J = gram(5, 3)
F = {(1, 1): Fr(1), (2, 0): Fr(7, 10), (0, 1): Fr(1, 14), (1, 0): Fr(-3, 14)}
c = [F.get(b, Fr(0)) for b in B]
I5, J5 = quad(c, I), quad(c, J)
print("I_5 =", I5, "  sum J =", J5, "  ratio =", J5 / I5, "=", float(J5 / I5))

def best_float(k, d):
    B, I, J = gram(k, d)
    A = np.array(I, dtype=float); M = np.array(J, dtype=float)
    L = np.linalg.cholesky(A); Li = np.linalg.inv(L)
    return np.linalg.eigvalsh(Li @ M @ Li.T)[-1]

print([round(best_float(5, d), 6) for d in range(1, 9)])
for k in range(2, 11):
    print(f"k={k:2d}  degree 6: {best_float(k, 6):.5f}   upper bound k log k/(k-1) = {k*math.log(k)/(k-1):.5f}")

The exact values match Maynard's: $I_5=\frac{29509}{1222452000}$, $\sum J=\frac{40493}{838252800}$, ratio $\frac{1417255}{708216}=2.00116$. Optimizing, degree $1$ gives $1.9508$, degree $2$ gives $1.9906$, and degree $3$ already $2.0027>2$; the values then settle at $2.0041$. So the Elliott--Halberstam conjecture would give gaps at most $12$ with a cubic polynomial in two symmetric functions. The table for $k\le10$ sits just below the upper bound of Exercise 9.5 for $k=2$ ($1.38593$ against $1.38629$) and drifts away from it as $k$ grows, as Figure 9.2 shows: at $k=10$, $2.526$ against $2.558$.

### P9.3

In [ ]:
def best_mp(k, d, dps):
    mp.mp.dps = dps
    B, I, J = gram(k, d)
    A = mp.matrix([[mp.mpf(x.numerator) / x.denominator for x in r] for r in I])
    M = mp.matrix([[mp.mpf(x.numerator) / x.denominator for x in r] for r in J])
    L = mp.cholesky(A); Li = mp.inverse(L)
    E, Q = mp.eigsy(Li * M * Li.T)
    i = max(range(len(E)), key=lambda j: E[j])
    v = Li.T * Q[:, i]
    return E[i], [v[j] for j in range(len(B))], B, I, J

t0 = time.time()
lam, v, B, I, J = best_mp(105, 11, 400)
print(len(B), "monomials; top eigenvalue", mp.nstr(lam, 12), f"({time.time()-t0:.1f}s)")
big = max(abs(x) for x in v)
for digits in [10, 15, 20, 25, 30]:
    c = [Fr(int(mp.nint(x / big * 10 ** digits)), 10 ** digits) for x in v]
    R = quad(c, J) / quad(c, I)
    print(f"{digits:2d} digits: exact ratio = {float(R):.10f}   > 4: {R > 4}")

With $400$ digits of working precision the top eigenvalue is $4.0020697619\ldots$, Maynard's value. Rounded to $10$ or $15$ significant digits the eigenvector gives ratios below $4$ ($1.76$ and $3.993$); from $20$ digits on, the exact rational ratio is $4.00206976\ldots>4$. That is a proof: the exact computation uses only Lemma 9.3, rational arithmetic, and the coefficients we chose, however we chose them. The need for so many digits in the *coefficients* is the same ill-conditioning that P9.5 measures.

### P9.4

In [ ]:
def product_bound(k, A, T, n=4000):
    c = 1 - 1 / (1 + A * T)
    gamma = c / A
    u = rng.random((n, k - 1))
    X = (1 / (1 - u * c) - 1) / A          # inverse CDF of the density g^2/gamma on [0,T]
    P = (X.sum(axis=1) <= k - T).mean()
    # inner integral is log(1+AT)/(kA); with Maynard's T this factor is 1
    return (math.log(1 + A * T) / A) ** 2 * P / gamma

def maynard_choice(k):
    A = math.log(k) - 2 * math.log(math.log(k)); return A, (math.exp(A) - 1) / A

def optimize(k):
    best = (0, None, None)
    for A in np.linspace(0.5, math.log(k), 25):
        for T in np.linspace(0.5, min(k / 2, 3 * math.exp(A) / A), 25):
            v = product_bound(k, A, T, n=400)
            if v > best[0]: best = (v, A, T)
    v, A, T = best
    return product_bound(k, A, T, n=4000), A, T

for k in [50, 105, 200, 250, 1000, 5000]:
    A, T = maynard_choice(k)
    mb = product_bound(k, A, T) if A > 0 else float('nan')
    v, Ao, To = optimize(k)
    print(f"k={k:6d}  Maynard's A,T: {mb:.3f}   optimized: {v:.3f} (A={Ao:.2f}, T={To:.1f})   log k - 2 loglog k - 2 = {math.log(k)-2*math.log(math.log(k))-2:.3f}")

For a general $T$ the inner integral is $\frac{\log(1+AT)}{kA}$, so the bound is $\bigl(\frac{\log(1+AT)}A\bigr)^2\frac{\mathbb P}\gamma$; Maynard's $T$ makes the first factor $1$. The product functions are well below the polynomials for small $k$: about $2.81$ at $k=50$ and $3.28$ at $k=105$, against $3.66$ and $4.002$ (the simulation has a sampling error of a few thousandths, and the search over a grid of $(A,T)$ biases the first estimate upwards, which is why each optimum is re-estimated with fresh samples). Optimized, they reach $4$ only at about $k=250$ ($3.92$ at $k=200$, $4.00$ at $k=250$, $5.36$ at $k=1000$), while Maynard's own $A,T$, chosen to make the proof easy, give only $1.99$ at $k=105$ and $3.20$ at $k=1000$. The asymptotic statement $\log k-2\log\log k-2$ is weaker still: it becomes positive only around $k\approx210$. This is the usual division of labour: the clean construction proves the theorem for every $m$, and a computer search over a richer family produces the good numbers for small $m$. The optimized products stay below the upper bound $\frac{k\log k}{k-1}$ ($6.91$ at $k=1000$), as they must.

### P9.5

In [ ]:
B, I, J = gram(105, 11)
A = np.array(I, dtype=float); M = np.array(J, dtype=float)
try:
    L = np.linalg.cholesky(A)
    Li = np.linalg.inv(L)
    print("float64 Cholesky succeeded; top eigenvalue", np.linalg.eigvalsh(Li @ M @ Li.T)[-1])
except np.linalg.LinAlgError as e:
    print("float64 Cholesky failed:", e)
w = np.linalg.eigvalsh(A)
print("float64 eigenvalues of I: smallest", w[0], " largest", w[-1])

mp.mp.dps = 400
for k in [5, 20, 50, 105]:
    B, I, J = gram(k, 11)
    Am = mp.matrix([[mp.mpf(x.numerator) / x.denominator for x in r] for r in I])
    ev = mp.eigsy(Am, eigvals_only=True)
    print(f"k={k:3d}: condition number of I = {mp.nstr(max(ev)/min(ev), 5)}")

The Gram matrix of the $42$ monomials is extremely ill-conditioned (condition numbers in the output above): the monomials $(1-P_1)^aP_2^b$ are nearly linearly dependent on the simplex, because for large $k$ a typical point has $P_2$ tiny and $1-P_1$ concentrated, so many basis functions look almost alike. In double precision, with $16$ significant digits, the smallest eigenvalues of $I$ drown in rounding errors: `numpy` reports a negative eigenvalue, $-2\cdot10^{-195}$, impossible for a Gram matrix. Here Cholesky happens to go through and the top eigenvalue comes out as $4.00206974$, right to eight digits, but with a condition number of $3\cdot10^{32}$ nothing guaranteed even the first digit, and a slightly different basis or $k$ makes Cholesky fail outright. Three remedies, all standard in numerical linear algebra: (i) work in higher precision, as here; (ii) choose a better-conditioned basis, for instance polynomials orthogonal for the measure on the simplex; (iii) whatever the method, *certify* the answer by an exact computation, as in P9.3. Remedy (iii) is what turns a numerical experiment into a proof: the eigenvalue solver only has to find good coefficients, and the inequality $\frac{\sum J}{I}>4$ is then checked with integers.